# 🇰🇭 Fine-tune Khmer LLM — QLoRA on Llama-SEA-LION-v3-8B-IT

Notebook នេះធ្វើ **QLoRA fine-tuning** (parameter-efficient, 4-bit) លើ model `aisingapore/Llama-SEA-LION-v3-8B-IT`
ដោយប្រើ dataset `saillab/alpaca-khmer-cleaned` ។ រចនាឡើងឲ្យដំណើរការលើ **GPU ឥតគិតថ្លៃរបស់ Colab ឬ Kaggle (T4, 16GB VRAM)**។

**ហេតុអ្វីជ្រើសរើស model នេះ?** SEA-LION ត្រូវបាន continued-pretrain ជាមួយភាសាខ្មែរផ្ទាល់ (ក្នុងចំណោមភាសាអាស៊ីអាគ្នេយ៍ ១១)
ដូច្នេះ base knowledge របស់ភាសាខ្មែរខ្លាំងជាង Llama/Qwen/Mistral ធម្មតាច្រើន — ត្រូវការ fine-tuning data តិចជាង
ដើម្បីទទួលបានលទ្ធផលល្អ។

**រយៈពេលប៉ាន់ស្មាន**: ~30-60 នាទី សម្រាប់ 3 epochs (អាស្រ័យលើ GPU ដែលបានចាត់ចែង និងទំហំ dataset subset)។

## ជំហានទី ០ — បើក GPU

- **Google Colab**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU**
- **Kaggle**: Notebook settings (រូបភ្នែក⚙️ខាងស្តាំ) → Accelerator → **GPU T4 x2** ឬ **P100**

ត្រូវប្រាកដថា GPU បើកមុននឹង run cell ខាងក្រោម — បើមិនដូច្នេះទេ training នឹងយឺតខ្លាំង ឬ error ។

In [ ]:
# ដំឡើង library ចាំបាច់ (~2-3 នាទី)
!pip install -q -U "transformers>=4.46" "trl>=0.24" peft bitsandbytes accelerate datasets sentencepiece

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("⚠️  គ្មាន GPU រកឃើញទេ — សូមត្រឡប់ទៅជំហានទី ០ ដើម្បីបើក GPU accelerator សិន")

## ជំហានទី ០.៥ — Login Hugging Face (បើ model gated)

`Llama-SEA-LION-v3` ផ្អែកលើ Llama ដូច្នេះ Hugging Face អាចទាមទារឲ្យអ្នក **ទទួលយក license**
និង **login ដោយ token** មុននឹងទាញ weights បាន។ បើ cell ទាញ model ខាងក្រោមចេញ error `401/403 gated`,
រត់ cell នេះ (បន្ថែម token ពី https://huggingface.co/settings/tokens ) រួចទៅទំព័រ model ចុច *Agree*៖

In [ ]:
# ដោះ comment ខាងក្រោមបើ model gated (បើមិន gated អាចរំលង cell នេះ)
# from huggingface_hub import login
# login()  # បិទភ្ជាប់ HF token នៅពេលសួរ

## Config
កែតម្លៃទាំងនេះបានតាមតម្រូវការ — នេះជាកន្លែងតែមួយគត់ដែលអ្នកប្រហែលជាត្រូវកែ។

In [ ]:
MODEL_ID = "aisingapore/Llama-SEA-LION-v3-8B-IT"   # base model (SEA-LION, គាំទ្រខ្មែរ)
DATASET_ID = "saillab/alpaca-khmer-cleaned"          # Khmer instruction dataset
OUTPUT_DIR = "./sealion-khmer-lora"
MAX_LENGTH = 1024        # កាត់បន្ថយបើ OOM (out of memory)
NUM_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8    # effective batch size = 2 x 8 = 16
LEARNING_RATE = 2e-4

# ដាក់ចំនួន example កំណត់សម្រាប់សាកល្បងលឿន (None = ប្រើទាំងអស់)
# ចាប់ផ្តើមដោយលេខតូច (ឧ. 500) ដើម្បីប្រាកដថា pipeline ដំណើរការត្រឹមត្រូវ មុននឹង train ពេញ dataset
MAX_TRAIN_EXAMPLES = 500

## ជំហានទី ១ — ទាញយក dataset ភាសាខ្មែរ

In [ ]:
raw_dataset = load_dataset(DATASET_ID)
print(raw_dataset)
print("\n--- ឧទាហរណ៍ទី ០ ---")
print(raw_dataset["train"][0])

⚠️ **ពិនិត្យលទ្ធផលខាងលើ**៖ dataset នេះគួរមាន field ឈ្មោះ `instruction`, `input`, `output` (Alpaca format ស្តង់ដារ)។
បើ column name ខុសពីនេះ សូមកែ function `format_example` នៅជំហានទី ៣ ខាងក្រោម ឲ្យត្រូវនឹង field ពិតប្រាកដ។

## ជំហានទី ២ — ទាញយក tokenizer + model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # float16 (មិនមែន bfloat16) ព្រោះ T4/P100 មិនគាំទ្រ bf16 ពេញលេញ
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # SFT ត្រូវ pad ខាងស្តាំ ដើម្បីជៀសបញ្ហា loss ជាមួយ fp16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
print("Model loaded ✅")

## ជំហានទី ៣ — Format data ជា chat template របស់ model

ប្រើ `tokenizer.apply_chat_template` ដើម្បីធានាថា format ត្រូវគ្នានឹងរបៀបដែល model ត្រូវបាន instruction-tune ពីដើម
(ប្រសើរជាងច្នៃប្រឌិត template ផ្ទាល់ខ្លួន)។

In [ ]:
def format_example(example):
    instruction = (example.get("instruction") or "").strip()
    extra_input = (example.get("input") or "").strip()
    output = (example.get("output") or "").strip()

    user_content = instruction
    if extra_input:
        user_content = f"{instruction}\n\n{extra_input}"

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}


train_split = raw_dataset["train"]
if MAX_TRAIN_EXAMPLES:
    train_split = train_split.select(range(min(MAX_TRAIN_EXAMPLES, len(train_split))))

formatted_dataset = train_split.map(
    format_example,
    remove_columns=train_split.column_names,
)

print("--- ឧទាហរណ៍បន្ទាប់ពី format ---")
print(formatted_dataset[0]["text"])

## ជំហានទី ៤ — រៀបចំ LoRA (parameter-efficient fine-tuning)

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## ជំហានទី ៥ — កំណត់ Training arguments និង Trainer

⚠️ **ចំណាំអំពី `trl` library**: package នេះកែប្រែ API រហ័សណាស់។ បើអ្នកជួប `TypeError` អំពី `tokenizer=`,
`dataset_text_field=`, ឬ `max_seq_length=` មិនស្គាល់ — នោះមានន័យថា version ថ្មីបានផ្លាស់ប្តូរឈ្មោះ argument ។
កូដខាងក្រោមប្រើ syntax បច្ចុប្បន្ន (SFTConfig ផ្ទុក dataset/training arguments ទាំងអស់, `processing_class` ជំនួស `tokenizer`)។

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    packing=False,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=True,                      # T4/P100: fp16, មិនមែន bf16
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
)

## ជំហានទី ៦ — ចាប់ផ្តើម Train 🚀

In [ ]:
trainer.train()

## ជំហានទី ៧ — រក្សាទុក LoRA adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"រក្សាទុករួចរាល់នៅ {OUTPUT_DIR}")

## ជំហានទី ៨ — សាកល្បង model ថ្មី

In [ ]:
model.config.use_cache = True   # បើក cache វិញសម្រាប់ generation លឿន (បានបិទពេល train)
model.eval()
test_messages = [{"role": "user", "content": "សូមណែនាំរបៀបធ្វើម្ហូបខ្មែរសាមញ្ញមួយមុខ"}]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(inputs, max_new_tokens=256, do_sample=True, temperature=0.7)

print(tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=True))

## ជំហានទី ៩ — ប្រើ adapter នេះក្នុង A2I

Training ខាងលើ រក្សាទុកតែ **LoRA adapter** (តូច ~១០០MB) — មិនមែន model ពេញ ៨B ទេ។
ដើម្បីប្រើក្នុង A2I មានពីរផ្លូវ៖

**ក. រក្សាទុក adapter ជាមុនសិន** (កុំឲ្យ Colab លុបចោល)៖

```python
# ទៅ Google Drive
from google.colab import drive; drive.mount('/content/drive')
!cp -r ./sealion-khmer-lora /content/drive/MyDrive/
# ឬ ទៅ Hugging Face Hub
# model.push_to_hub("your-username/sealion-khmer-lora")
```

**ខ. បំប្លែងទៅ GGUF សម្រាប់ A2I Core / llama.cpp**

⚠️ *កុំ* `merge_and_unload()` លើ model 4-bit ដែលទើប train — វា merge មិនស្អាតទេ។
ត្រូវ **reload base ជា fp16** សិន រួច merge។ 8B fp16 ត្រូវ ~16GB — លើសពី RAM/VRAM ឥតគិតថ្លៃរបស់ Colab
ដូច្នេះ merge នៅលើម៉ាស៊ីនធំជាង (A100 ឬ local ដែលមាន RAM គ្រប់)៖

```python
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained("sealion-khmer-merged")
tokenizer.save_pretrained("sealion-khmer-merged")

# → GGUF (q4_k_m ល្អសម្រាប់ CPU/A2I Core)
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py sealion-khmer-merged \
    --outfile sealion-khmer.gguf --outtype q8_0
!./llama.cpp/llama-quantize sealion-khmer.gguf sealion-khmer-q4_k_m.gguf q4_k_m
```

បន្ទាប់មក ដាក់ file GGUF នៅ `a2i-core/models/model.gguf` រួច `./run.sh` — A2I Core នឹងបម្រើ
model ខ្មែរថ្មីរបស់អ្នកតាម API ដដែល (មើល `a2i-train/README.md`)។

**គន្លឹះ**: បើ merge 8B ពិបាកពេក អ្នកអាចទុក adapter ដាច់ដោយឡែក ហើយ llama.cpp គាំទ្រ GGUF LoRA
(`convert_lora_to_gguf.py` + flag `--lora` ពេល serve) — base GGUF មួយ + adapter តូចមួយ។

## ជំហានបន្ទាប់

- **រក្សាទុកអចិន្ត្រៃយ៍**: Colab/Kaggle លុប file ចោលពេល session ចប់ — មើលជំហានទី ៩.ក ខាងលើ។
- **Train ពេញ dataset**: កែ `MAX_TRAIN_EXAMPLES = None` នៅ Config ខាងលើ ពេលដែល pipeline ដំណើរការត្រឹមត្រូវហើយ
  (នេះនឹងចំណាយពេលយូរជាង — តាមដាន GPU-hour quota របស់ Colab/Kaggle)។
- **ល្បឿនលឿនជាង**: library `unsloth` អាចបង្កើនល្បឿន 1.5-2x និងកាត់បន្ថយ VRAM បន្ថែមទៀត — ពិចារណាប្រើពេលដែល
  pipeline មូលដ្ឋាននេះដំណើរការល្អហើយ។
- **Model ធំជាង**: បើមាន GPU ធំជាង (A100 ឬច្រើនជាង 24GB VRAM) អាចប្តូរទៅ `aisingapore/Gemma-SEA-LION-v4-27B-IT`
  សម្រាប់ performance ខ្ពស់ជាង។